In [474]:
import pandas as pd
import random as rd

In [475]:
def escolhe_pacote(name, show = False):
    data = pd.read_csv(f'packs\{name}.csv')
    if show == True:
        for index, row in data.iterrows():
            print(f'{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]}')


In [476]:
def rodar_pacote(name):
    data = pd.read_csv(f'packs\{name}.csv')
    pacote = []
    box = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity'])

    commons = data[data['rarity'] == 'C']
    c_tiradas = commons.sample(n=4).to_dict('records')
    pacote.extend(c_tiradas)

    luck = rd.randint(1, 100)

    if luck <= 70:
        raridade = 'R'
    elif luck <= 90:
        raridade = 'RR'
    elif luck <= 99:
        raridade = 'RRR'
    else:
        raridade = 'SP'

    raras = data[data['rarity'] == raridade]
    r_tirada = raras.sample(n=1).to_dict('records')
    pacote.extend(r_tirada)

    pacote = pd.DataFrame(pacote)
    
    return pacote

In [477]:
def atualizar_save(box, pacote):
    #Quero colocar os valores do pacote na box e se houver repetidos, qtt += 1
    
    for index, row in pacote.iterrows():
        card = box[(box['set'] == row['set']) & (box['id'] == row['id'])]
        if not card.empty:
            box.loc[card.index, 'qtt'] += 1
        else:
            new_card = row.to_dict()
            new_card['qtt'] = 1
            #box não tem append, então vou criar um novo dataframe com a nova linha e concatenar com a box
            new_card_df = pd.DataFrame([new_card])
            box = pd.concat([box, new_card_df], ignore_index=True)
           
    return box

In [478]:
def rodar_box(name, qtt):
    box = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])
    for i in range(qtt):
        pacote = rodar_pacote(name)
        box = atualizar_save(box, pacote)
    
    try:
        save = pd.read_csv('save.csv')
    except:
        save = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])
    box.sort_values(by=['id'], inplace=True)
    box.to_csv('last_box.csv', index=False)
    save = atualizar_save(save, box)
    save.sort_values(by=['id'], inplace=True)
    save.to_csv('save.csv', index=False)
    return box

In [479]:
r = rodar_box('BT01', 10)
colecao = pd.read_csv('save.csv')
colecao

,set,id,name,grade,clan,type,rarity,qtt
0,BT01,1,"King of Knights, Alfred",3,Royal Paladin,Normal Unit,RRR,2
1,BT01,2,Blaster Blade,2,Royal Paladin,Normal Unit,RRR,3
2,BT01,3,Barcgal,0,Royal Paladin,Normal Unit,RRR,2
3,BT01,4,Dragonic Overlord,3,Kagero,Normal Unit,RRR,1
4,BT01,5,"Embodiment of Victory, Aleph",3,Kagero,Normal Unit,RRR,3
...,...,...,...,...,...,...,...,...
73,BT01,76,"Redshoe, Milly",0,Spike Brothers,Normal Unit,C,9
74,BT01,77,"Dandy Guy, Romario",1,Granblue,Normal Unit,C,9
75,BT01,78,Guiding Zombie,0,Granblue,Normal Unit,C,5
76,BT01,79,Karma Queen,1,Megacolony,Normal Unit,C,7
